# Week6_ex2 - 3-Slot Linear Motor, Single-Phase Current, Moving Magnet Array

Builds on `Week6_ex1.ipynb`'s stationary-mover setup by giving the mover actual velocity, synchronized with the current frequency, over one electrical period. Transient is blocked on the Student license here too, so this uses the same `t_sample` trick as ex1, but now t_sample also drives the mover's position, not just the current.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

In [ ]:
## Variables ##

# geometry

depth = 40     # stack length of the model
num_pole = 10    # number of magnet poles
m_w = 20   # magnet width
m_h = 5   # magnet height
gap_w = m_w * 0.3   # spacing between magnets
my_th = 15   # magnet plate thickness
num_slot = 3   # number of slots
airgap = 1   # air gap length
t_w = 6   # tooth width
t_g = (52 / 3 - 6) / 2   # tooth spacing
J_rated = 3   # coil region current density
cy_th = 15   # yoke thickness
c_w = 5   # coil width
c_h = 15   # coil height
t_h = c_h   # tooth height
my_w = (m_w * num_pole + gap_w * (num_pole - 1)) / 3 * 4   # magnet plate width


# current settings

turn_c = 100    # coil turns

area = c_w * c_h   # terminal area

current_rated = J_rated * area / turn_c    # phase rated current

gamma = math.pi * 0   # current angle mismatch
theta_elec_offset = math.pi / 2 + gamma   # id=0 control + mismatch

freq_elec = 10      # electrical frequency

# NOTE: the original Transient version drives this with the solver's built-in
# "Time" variable. Magnetostatic has no time axis, so this uses the
# "t_sample" design variable instead -- each parametric step evaluates the
# expression at a fixed t_sample and solves a static snapshot.
current_a = f"{current_rated}*cos(2*pi*{freq_elec}*t_sample + {theta_elec_offset})"

current_b = 0

current_c = 0


# mover displacement

movement = True   # True: +x direction, False: -x direction
move_direction = 1 if movement else -1

veolcity_stator = 2*(m_w+gap_w)/1000/(1/freq_elec)   # average speed over one electrical period (m/s)
# reused below as the position-sweep rate: displacement = veolcity_stator * t_sample,
# since Magnetostatic doesn't support a real motion boundary like Transient's band

In [ ]:
# 1. Launch AEDT
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. Turn off autosave
DT.disable_autosave()

# 3. Create project and set solution type
sol_type = "Magnetostatic"
M2D = ansys.aedt.core.maxwell.Maxwell2d(solution_type=sol_type)

# 4. odesign object for script-recording style calls
oDesign = M2D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week6_ex2"   # matches this exercise's folder name

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week6_ex2"

In [ ]:
## Save project and apply design name ##

proj = M2D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M2D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M2D.save_project()

In [ ]:
## Design variable definitions ##

M2D["depth"] = f"{depth}mm"
M2D["num_pole"] = f"{num_pole}"
M2D["m_w"] = f"{m_w}mm"
M2D["m_h"] = f"{m_h}mm"
M2D["gap_w"] = f"{gap_w}mm"
M2D["my_th"] = f"{my_th}mm"
M2D["num_slot"] = f"{num_slot}"
M2D["airgap"] = f"{airgap}mm"
M2D["t_w"] = f"{t_w}mm"
M2D["t_g"] = f"{t_g}mm"
M2D["J_rated"] = f"{J_rated}"
M2D["cy_th"] = f"{cy_th}mm"
M2D["c_w"] = f"{c_w}mm"
M2D["c_h"] = f"{c_h}mm"
M2D["t_h"] = f"{t_h}mm"
M2D["my_w"] = f"{my_w}mm"
M2D["turn_c"] = f"{turn_c}"
M2D["current_rated"] = f"{current_rated}"
M2D["gamma"] = f"{gamma}"
M2D["theta_elec_offset"] = f"{theta_elec_offset}"
M2D["freq_elec"] = f"{freq_elec}"

# position-sweep variables -- Magnetostatic has no real motion boundary,
# so the mover's position is driven by t_sample instead
M2D["veolcity_stator"] = f"{veolcity_stator}m_per_sec"
M2D["t_sample"] = "0s"

In [ ]:
## Analyze Setup ##

# model depth
M2D.model_depth = "depth"

# Magnetostatic setup -- use create_setup() directly here, not the raw
# InsertSetup script-recording pattern (that pattern is only needed for
# EddyCurrent, where AEDT 2025 R2's internal "AC Magnetic" rename breaks
# create_setup()).
my_setup = M2D.create_setup(name="Setup1")
my_setup.props["MaximumPasses"] = 10
my_setup.props["PercentRefinement"] = 30
my_setup.update()

In [ ]:
## Material definitions ##

# steel
steel = M2D.materials.duplicate_material("steel_1008", name="steel")
steel.stacking_type = "Lamination"      # lamination to suppress eddy current loss
steel.stacking_factor = 0.95
steel.stacking_direction = "V(3)"

# magnet, N pole facing +z
magnet_n = M2D.materials.duplicate_material("NdFe35", name="magnet_n")
magnet_n.get_magnetic_coercivity()  # check magnetization
magnet_n.set_magnetic_coercivity(value=-890000, x=0, y=1, z=0)  # magnetization direction (+z)

# magnet, S pole facing -z
magnet_s = M2D.materials.duplicate_material("NdFe35", name="magnet_s")
magnet_s.get_magnetic_coercivity()  # check magnetization
magnet_s.set_magnetic_coercivity(value=-890000, x=0, y=-1, z=0)  # magnetization direction (-z)

# coil copper
copper_c = M2D.materials.duplicate_material("copper", name="copper_c")

In [ ]:
## Draw geometry and assign boundaries ##

# draw the magnet plate (substrate)

origin = ["-my_w/2", "-my_th/2"]
sizes = ["my_w", "my_th"]
substrate = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name='substrate', material=steel, non_model=False)

# draw the magnets

origin = [0, "-m_h/2"]
sizes = ["m_w", "m_h"]
magnet_1 = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="magnet_1", material=magnet_s, non_model=False)

vector = ["2*(m_w+gap_w)", 0]
M2D.modeler.duplicate_along_line(assignment=magnet_1, vector=vector, clones=num_pole/2, attach=True, is_3d_comp=False, duplicate_assignment=True)   # duplicate

magnet_1.color = (0, 0, 255)


origin = ["gap_w+m_w", "-m_h/2"]
sizes = ["m_w", "m_h"]
magnet_2 = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="magnet_2", material=magnet_n, non_model=False)

vector = ["2*(m_w+gap_w)", 0]
M2D.modeler.duplicate_along_line(assignment=magnet_2, vector=vector, clones=num_pole/2, attach=True, is_3d_comp=False, duplicate_assignment=True)   # duplicate

magnet_2.color = (255, 0, 0)


vector = ["-( m_w*num_pole+gap_w*(num_pole-1) )/2", "my_th/2+m_h/2"]
M2D.modeler.move(assignment=[magnet_1, magnet_2], vector=vector)    # move into place


M2D.modeler.separate_bodies(assignment=magnet_1, create_group=False)
magnet_1_1 = M2D.modeler.object_list[-4]
magnet_1_2 = M2D.modeler.object_list[-3]
magnet_1_3 = M2D.modeler.object_list[-2]
magnet_1_4 = M2D.modeler.object_list[-1]
M2D.modeler.separate_bodies(assignment=magnet_2, create_group=False)
magnet_2_1 = M2D.modeler.object_list[-4]
magnet_2_2 = M2D.modeler.object_list[-3]
magnet_2_3 = M2D.modeler.object_list[-2]
magnet_2_4 = M2D.modeler.object_list[-1]


# draw the slots

origin = [0, "-t_h/2"]
sizes = ["t_w", "t_h"]
slot_a = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="slot_a", material="air", non_model=False)


vector=["t_w+2*t_g", 0]
M2D.modeler.duplicate_along_line(assignment=slot_a, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
slot_b = M2D.modeler.object_list[-1]
slot_b.name = "slot_b"

M2D.modeler.duplicate_along_line(assignment=slot_b, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
slot_c = M2D.modeler.object_list[-1]
slot_c.name = "slot_c"


# draw the coils

origin = ["-c_w", "-t_h/2"]
sizes = ["t_w+2*c_w", "c_h"]
coil_a = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="coil_a", material=copper_c, non_model=False)


M2D.modeler.subtract(blank_list=coil_a, tool_list=slot_a, keep_originals=True)  # (coil)-(slot)


vector=["t_w+2*t_g", 0]
M2D.modeler.duplicate_along_line(assignment=coil_a, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
coil_b = M2D.modeler.object_list[-1]
coil_b.name = "coil_b"

M2D.modeler.subtract(blank_list=coil_b, tool_list=slot_b, keep_originals=True)  # (coil)-(slot)


vector=["t_w+2*t_g", 0]
M2D.modeler.duplicate_along_line(assignment=coil_b, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
coil_c = M2D.modeler.object_list[-1]
coil_c.name = "coil_c"

M2D.modeler.subtract(blank_list=coil_c, tool_list=slot_c, keep_originals=True)  # (coil)-(slot)


vector = ["-( 1.5*t_w+2*t_g )", 0]
M2D.modeler.move(assignment=[coil_a, coil_b, coil_c, slot_a, slot_b, slot_c], vector=vector)    # move into place


# draw the yoke

origin = ["-( (t_w+2*t_g)*3*1.3 )/2", "c_h/2"]
sizes = ["( (t_w+2*t_g)*3*1.3 )", "cy_th"]
stator = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="stator", material="air", non_model=False)


M2D.modeler.unite(assignment=[stator, slot_a, slot_b, slot_c], purge=False, keep_originals=False)

vector = [0, "my_th/2+m_h+airgap+c_h/2"]
M2D.modeler.move(assignment=[stator, coil_a, coil_b, coil_c], vector=vector)    # move into place


# align magnet plate

# vector = ["3*(gap_w+m_w)/2-t_w-2*t_g", 0]
# M2D.modeler.move(assignment=[substrate, magnet_1, magnet_1_1, magnet_1_2, magnet_1_3, magnet_1_4, magnet_2, magnet_2_1, magnet_2_2, magnet_2_3, magnet_2_4], vector=vector)    # a-phase / alpha-axis alignment
vector = ["-(3*(gap_w+m_w)/2-t_w-2*t_g)", 0]
M2D.modeler.move(assignment=[coil_a, coil_b, coil_c, stator], vector=vector)


# mover position sweep -- in the original Transient version, this displacement
# was handled by a translate-motion boundary on a "band" region. Magnetostatic
# doesn't support that, so the mover (substrate + magnets) is displaced
# directly using the t_sample design variable instead. Each parametric step
# moves the mover to its position at that instant and solves a static
# snapshot.
vector = [f"{move_direction}*veolcity_stator*t_sample", 0]
M2D.modeler.move(assignment=[substrate, magnet_1, magnet_1_1, magnet_1_2, magnet_1_3, magnet_1_4,
                              magnet_2, magnet_2_1, magnet_2_2, magnet_2_3, magnet_2_4], vector=vector)

# NOTE: region padding below is sized from my_w and was originally generous
# enough for the "band" motion boundary's own -80mm/+80mm travel range --
# haven't checked locally whether it's still generous enough once the mover
# is displaced directly like this. First thing to check during verification.


# draw the region

oEditor = oDesign.SetActiveEditor("3D Modeler")
oEditor.CreateRegion(
	[
		"NAME:RegionParameters",
		"+XPaddingType:=", "Absolute Position",
		"+XPadding:=", "my_w",
		"-XPaddingType:=", "Absolute Position",
		"-XPadding:=", "-my_w",
		"+YPaddingType:=", "Absolute Position",
		"+YPadding:=", "my_w",
		"-YPaddingType:=", "Absolute Position",
		"-YPadding:=", "-my_w",
		"+ZPaddingType:=", "Percentage Offset",
		"+ZPadding:=", "0",
		"-ZPaddingType:=", "Percentage Offset",
		"-ZPadding:=", "0",
		[
			"NAME:BoxForVirtualObjects",
			[
				"NAME:LowPoint", 
				1, 
				1, 
				1
			],
			[
				"NAME:HighPoint", 
				-1, 
				-1, 
				-1
			]
		]
	], 
	[
		"NAME:Attributes",
		"Name:=", "Region",
		"Flags:=", "Wireframe#",
		"Color:=", "(143 175 143)",
		"Transparency:=", 0,
		"PartCoordinateSystem:=", "Global",
		"UDMId:=", "",
		"MaterialValue:=", "\"vacuum\"",
		"SurfaceMaterialValue:=", "\"\"",
		"SolveInside:=", True,
		"ShellElement:=", False,
		"ShellElementThickness:=", "nan ",
		"ReferenceTemperature:=", "nan ",
		"IsMaterialEditable:=", True,
		"IsSurfaceMaterialEditable:=", True,
		"UseMaterialAppearance:=", False,
		"IsLightweight:=", False
	])

region = M2D.modeler.object_list[-1]

# assign balloon boundary
M2D.assign_balloon(assignment=region.edges, boundary=None)

In [ ]:
## Excitation ##

# create coil terminals

M2D.modeler.separate_bodies(assignment=[coil_a], create_group=False)   # split into coil cross-sections
coil_a_out = M2D.modeler.sheet_objects[-1]     # assign the new section
coil_a_in = coil_a

M2D.modeler.separate_bodies(assignment=[coil_b], create_group=False)   # split into coil cross-sections
coil_b_out = M2D.modeler.sheet_objects[-1]     # assign the new section
coil_b_in = coil_b

M2D.modeler.separate_bodies(assignment=[coil_c], create_group=False)   # split into coil cross-sections
coil_c_out = M2D.modeler.sheet_objects[-1]     # assign the new section
coil_c_in = coil_c


# create windings and add coil terminals

winding_a = M2D.assign_winding(assignment=None, winding_type='Current', is_solid=False, current=current_a, resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_a")    # create winding
winding_b = M2D.assign_winding(assignment=None, winding_type='Current', is_solid=False, current=current_b, resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_b")    # create winding
winding_c = M2D.assign_winding(assignment=None, winding_type='Current', is_solid=False, current=current_c, resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_c")    # create winding

coil_a_in_ter = M2D.assign_coil(assignment=coil_a_in, conductors_number=turn_c, polarity='Negative', name="coil_a_in_ter")   # assign coil
coil_a_out_ter = M2D.assign_coil(assignment=coil_a_out, conductors_number=turn_c, polarity='Positive', name="coil_a_out_ter")   # assign coil
coil_b_in_ter = M2D.assign_coil(assignment=coil_b_in, conductors_number=turn_c, polarity='Negative', name="coil_b_in_ter")   # assign coil
coil_b_out_ter = M2D.assign_coil(assignment=coil_b_out, conductors_number=turn_c, polarity='Positive', name="coil_b_out_ter")   # assign coil
coil_c_in_ter = M2D.assign_coil(assignment=coil_c_in, conductors_number=turn_c, polarity='Negative', name="coil_c_in_ter")   # assign coil
coil_c_out_ter = M2D.assign_coil(assignment=coil_c_out, conductors_number=turn_c, polarity='Positive', name="coil_c_out_ter")   # assign coil


M2D.add_winding_coils(assignment=winding_a.name, coils=[coil_a_in_ter.name, coil_a_out_ter.name])   # add coils to winding
M2D.add_winding_coils(assignment=winding_b.name, coils=[coil_b_in_ter.name, coil_b_out_ter.name])   # add coils to winding
M2D.add_winding_coils(assignment=winding_c.name, coils=[coil_c_in_ter.name, coil_c_out_ter.name])   # add coils to winding

In [ ]:
## Mesh settings ##

oModule = oDesign.GetModule("MeshSetup")
oModule.AssignLengthOp(
	[
		"NAME:Length1",
		"RefineInside:=", False,
		"Enabled:=", True,
		"Objects:=", ["magnet_1","magnet_2","magnet_1_Separate1","magnet_1_Separate2","magnet_1_Separate3","magnet_1_Separate4","magnet_2_Separate1","magnet_2_Separate2","magnet_2_Separate3","magnet_2_Separate4","coil_a","coil_b","coil_c","stator","coil_a_Separate1","coil_b_Separate1","coil_c_Separate1"],
		"RestrictElem:=", False,
		"NumMaxElem:=", "1000",
		"RestrictLength:=", True,
		"MaxLength:=", "0.5mm",
		"ApplyToInitialMesh:=", False,
		"IsGlobal:=", False
	])

In [ ]:
# generate mesh manually (optional)
# M2D.mesh.generate_mesh(my_setup.name)

In [ ]:
## Force parameter ##

M2D.assign_force(assignment=[coil_a_in, coil_a_out, coil_b_in, coil_b_out, coil_c_in, coil_c_out], coordinate_system='Global', is_virtual=True, force_name="force_virtual")

In [ ]:
## Parametric sweep over t_sample (position/time snapshots) ##

# t_sample is registered as a design variable above; assign_force must be
# registered before analyze(), so this sweep goes after the force cell.

sweep = M2D.parametrics.add("t_sample", 0, 1 / freq_elec, 0.005, variation_type="LinearStep")

# 0.005s step gives a reasonable number of snapshots across one electrical
# period without making the sweep too slow -- can shrink this once the
# per-point solve time is known locally.

sweep.props["ProdOptiSetupDataV2"]["SaveFields"] = True
sweep.update()

sweep.analyze()

In [ ]:
## Force vs t_sample data table ##

table_name = "force_table"

oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport(f"{table_name}", "Magnetostatic", "Rectangular Plot", "Setup1 : LastAdaptive", 
	[
		"Domain:=", "Sweep"
	], 
	[
		"t_sample:=", ["All"],
		"veolcity_stator:=", ["Nominal"],
		"depth:=", ["Nominal"],
		"num_pole:=", ["Nominal"],
		"m_w:=", ["Nominal"],
		"m_h:=", ["Nominal"],
		"gap_w:=", ["Nominal"],
		"my_th:=", ["Nominal"],
		"num_slot:=", ["Nominal"],
		"airgap:=", ["Nominal"],
		"t_w:=", ["Nominal"],
		"t_g:=", ["Nominal"],
		"J_rated:=", ["Nominal"],
		"cy_th:=", ["Nominal"],
		"c_w:=", ["Nominal"],
		"c_h:=", ["Nominal"],
		"t_h:=", ["Nominal"],
		"my_w:=", ["Nominal"],
		"turn_c:=", ["Nominal"],
		"current_rated:=", ["Nominal"],
		"gamma:=", ["Nominal"],
		"theta_elec_offset:=", ["Nominal"],
		"freq_elec:=", ["Nominal"]
	], 
	[
		"X Component:=", "t_sample",
		"Y Component:=", ["force_virtual.Force_x","force_virtual.Force_y"]
	])

# NOTE: report context uses "Setup1 : LastAdaptive" and sweeps t_sample over
# "All" instead of the original "Setup1 : Transient" / X=Time -- not yet
# verified locally that this context string is exactly right for a
# Magnetostatic parametric sweep report. First thing to check in batch
# verification.

csv_name = "Week6_ex2_force.csv"
csv_path = csv_name  # saved flat, next to the notebook -- same convention as Images/
oModule.ExportToFile(f"{table_name}", csv_path, False)

In [ ]:
## Plot force vs t_sample ##

pd_data = pd.read_csv(csv_path)

# rename columns
pd_data.columns = ["t_sample [s]", "Force_x [N]", "Force_y [N]"]

plt.figure(figsize=(12, 8))

plt.plot(pd_data["t_sample [s]"], pd_data["Force_x [N]"], marker='o', linestyle='--', linewidth=2, markersize=6, label="Force_x [N]")
plt.plot(pd_data["t_sample [s]"], pd_data["Force_y [N]"], marker='o', linestyle='--', linewidth=2, markersize=6, label="Force_y [N]")

plt.xlabel("t_sample [s]", fontsize=16)
plt.ylabel("Force [N]", fontsize=16)

# example bound, revisit once real values are in from a local run
plt.ylim(-4, 4)

plt.legend(fontsize=12)
plt.grid(True, linestyle='--', linewidth=0.5)

os.makedirs("Images", exist_ok=True)
plt.savefig("Images/Week6_ex2_force_vs_t_sample.png")
plt.show()

In [ ]:
## Current vs t_sample data table ##

table_name = "current_table"

oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport(f"{table_name}", "Magnetostatic", "Rectangular Plot", "Setup1 : LastAdaptive", 
	[
		"Domain:=", "Sweep"
	], 
	[
		"t_sample:=", ["All"],
		"veolcity_stator:=", ["Nominal"],
		"depth:=", ["Nominal"],
		"num_pole:=", ["Nominal"],
		"m_w:=", ["Nominal"],
		"m_h:=", ["Nominal"],
		"gap_w:=", ["Nominal"],
		"my_th:=", ["Nominal"],
		"num_slot:=", ["Nominal"],
		"airgap:=", ["Nominal"],
		"t_w:=", ["Nominal"],
		"t_g:=", ["Nominal"],
		"J_rated:=", ["Nominal"],
		"cy_th:=", ["Nominal"],
		"c_w:=", ["Nominal"],
		"c_h:=", ["Nominal"],
		"t_h:=", ["Nominal"],
		"my_w:=", ["Nominal"],
		"turn_c:=", ["Nominal"],
		"current_rated:=", ["Nominal"],
		"gamma:=", ["Nominal"],
		"theta_elec_offset:=", ["Nominal"],
		"freq_elec:=", ["Nominal"]
	], 
	[
		"X Component:=", "t_sample",
		"Y Component:=", ["InputCurrent(winding_a)","InputCurrent(winding_b)","InputCurrent(winding_c)"]
	])

csv_name = "Week6_ex2_current.csv"
csv_path = csv_name  # saved flat, next to the notebook -- same convention as Images/
oModule.ExportToFile(f"{table_name}", csv_path, False)

In [ ]:
## Plot current vs t_sample ##

pd_data = pd.read_csv(csv_path)

pd_data.columns = ["t_sample [s]", "current_a [A]", "current_b [A]", "current_c [A]"]

plt.figure(figsize=(12, 8))

plt.plot(pd_data["t_sample [s]"], pd_data["current_a [A]"], linestyle='-', linewidth=4, markersize=6, label="current_a [A]")
plt.plot(pd_data["t_sample [s]"], pd_data["current_b [A]"], linestyle='-', linewidth=4, markersize=6, label="current_b [A]")
plt.plot(pd_data["t_sample [s]"], pd_data["current_c [A]"], linestyle='--', linewidth=4, markersize=6, label="current_c [A]")

plt.xlabel("t_sample [s]", fontsize=16)
plt.ylabel("Current [A]", fontsize=16)

plt.legend(fontsize=12)
plt.grid(True, linestyle='--', linewidth=0.5)

os.makedirs("Images", exist_ok=True)
plt.savefig("Images/Week6_ex2_current_vs_t_sample.png")
plt.show()

In [ ]:
## Save project ##

M2D.save_project()